In [1]:
import pandas as pd
from pyecharts.charts import Line
import pyodbc
import datetime

In [2]:
data = input('请输入要查询的月份（2025-03）：')

mdb_file = r'\\192.168.0.118\金恒晟共享文档\ZKTIMEACCESS\test.MDB'
conn_str = (
    rf'DRIVER={{Microsoft Access Driver (*.mdb, *.accdb)}};'
    rf'DBQ={mdb_file};'
)

conn = pyodbc.connect(conn_str)

sql = """
SELECT 
    c.CHECKTIME AS 时间,
    U.[Name] AS 姓名,
    c.SENSORID AS 机器号,
    D.DEPTNAME AS 部门名称
FROM 
    (CHECKINOUT c 
INNER JOIN 
    USERINFO U ON c.USERID = U.USERID)
INNER JOIN
    DEPARTMENTS D ON U.DEFAULTDEPTID = D.DEPTID
WHERE 
    c.CHECKTIME > #2025-03-01 00:00:00#
    AND (c.SENSORID = '1' OR c.SENSORID = '11')
"""

df = pd.read_sql(sql, conn)

df = df.sort_values(['姓名','时间'])
df['考勤日期'] = df['时间'].dt.date
df['考勤时间'] = df['时间'].dt.time
df = df[['姓名','考勤日期','考勤时间','部门名称','机器号']]
df.to_excel('4.xlsx',index=False,)

请输入要查询的月份（2025-03）：2025-04


C:\Users\Administrator\AppData\Local\Temp\ipykernel_40372\106994497.py:28: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


In [3]:
df

,姓名,考勤日期,考勤时间,部门名称,机器号
21,丁现,2025-03-01,07:27:01,铝塑膜事业部,1
201,丁现,2025-03-01,11:22:03,铝塑膜事业部,1
252,丁现,2025-03-01,11:34:23,铝塑膜事业部,1
968,丁现,2025-03-01,18:04:38,铝塑膜事业部,1
1693,丁现,2025-03-03,07:24:27,铝塑膜事业部,1
...,...,...,...,...,...
49066,龙志勇,2025-04-09,06:04:09,设备部,1
50614,龙志勇,2025-04-09,21:36:53,设备部,1
50792,龙志勇,2025-04-10,06:04:03,设备部,1
52636,龙志勇,2025-04-10,21:48:34,设备部,11


In [5]:
name_df = df.groupby('姓名')['考勤日期'].size().sort_values(ascending=False)

In [35]:
newdf = df[df['姓名'] == '龙志勇'].groupby('考勤日期').size()

In [44]:
result = []
for i in newdf.index:
    if newdf[i]<2:
        result.append([i,newdf[i]])
    elif newdf[i] > 2:
        result.append([i,newdf[i]])
        

In [46]:
resu

考勤日期
2025-03-01    1
2025-03-02    1
2025-03-03    1
2025-03-04    2
2025-03-05    2
2025-03-06    2
2025-03-07    2
2025-03-08    2
2025-03-09    2
2025-03-10    2
2025-03-11    2
2025-03-12    2
2025-03-13    2
2025-03-14    2
2025-03-15    2
2025-03-16    1
2025-03-17    2
2025-03-18    2
2025-03-19    2
2025-03-20    2
2025-03-21    2
2025-03-23    2
2025-03-24    3
2025-03-25    2
2025-03-26    2
2025-03-27    2
2025-03-28    2
2025-03-30    3
2025-03-31    3
2025-04-01    1
2025-04-02    2
2025-04-03    2
2025-04-04    1
2025-04-07    1
2025-04-08    2
2025-04-09    2
2025-04-10    2
2025-04-11    1
dtype: int64

In [ ]:
df.loc[(df['姓名'] == '楚红均')].dtypes

In [ ]:
time_data = df[(df['考勤日期'] == datetime.date(2025,3,1)) & (df['姓名'] == '楚红均')]

In [ ]:
time_data

In [ ]:
df['时间戳'] = pd.to_datetime(df['考勤日期'].astype('str') + ' ' + df['考勤时间'].astype('str'))

In [ ]:
df

In [ ]:
def filter_record(group):
    result = [group.iloc[0]]
    
    for i in range(1,len(group)):
        
        current = group.iloc[i]
        previous = result[-1]
        
        if current['时间戳'].hour != previous['时间戳'].hour:
            result.append(current)
            
        else:
            minutes_diff = (current['时间戳'] - previous['时间戳']).total_seconds() /60
            if minutes_diff > 1:
                result.append(current)
    return pd.DataFrame(result)
    

In [ ]:
filter_group = df.groupby(['姓名','考勤日期'],group_keys=False).apply(filter_record)

In [ ]:
filter_group.drop('时间戳',inplace=True,axis=1)

In [ ]:
filter_group.to_excel('filter_group.xlsx',index=False)